# 08 — Docker Compose, Observability, Performance, and Production Readiness

Goal: operate full local deployment stack like production mini-environment.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pipeline').exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / 'pipeline').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)

Project root: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/MLOps, UI, and Deployment/Deploy a Machine Learning Model with Docker


## 8.1 Compose Orchestration and Networking
**Definition:** Compose defines multi-service topology and shared network contracts in one file.

**Theory:** Explain mechanism, assumptions, and where this fits in deployment lifecycle.

**Motivation:** Why this matters for reliability, reproducibility, and operations.

**Real-World Example:** API + Prometheus + Grafana run on same bridge network for metric scraping and dashboards.

**Visual Explanation:** See figure/code cell below.

**Code Explanation:** Code cell demonstrates concrete implementation details.

**Best Practices:** Treat compose file as executable infrastructure documentation.

**Common Mistakes:** Hardcoding host-local assumptions instead of service DNS names.

In [2]:
import subprocess

proc = subprocess.run(['docker', 'compose', 'config'], capture_output=True, text=True)
if proc.returncode == 0:
    print('docker compose config valid')
    print('\n'.join(proc.stdout.splitlines()[:40]))
else:
    print('docker compose config failed (expected if docker daemon unavailable)')
    print(proc.stderr.strip())

docker compose config valid
name: deployamachinelearningmodelwithdocker
services:
  grafana:
    depends_on:
      prometheus:
        condition: service_started
        required: true
    environment:
      GF_SECURITY_ADMIN_PASSWORD: admin
      GF_SECURITY_ADMIN_USER: admin
      GF_USERS_ALLOW_SIGN_UP: "false"
    image: grafana/grafana:11.1.4
    networks:
      mlops-net: null
    ports:
      - mode: ingress
        target: 3000
        published: "3000"
        protocol: tcp
    volumes:
      - type: volume
        source: grafana_data
        target: /var/lib/grafana
        volume: {}
      - type: bind
        source: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/MLOps, UI, and Deployment/Deploy a Machine Learning Model with Docker/monitoring/grafana/provisioning
        target: /etc/grafana/provisioning
        read_only: true
        bind: {}
      - type: bind
        source: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/MLOps, UI, and Deployment/Deploy a

## 8.2 Monitoring, Logging, Security, and Checklist
**Definition:** Operational readiness combines telemetry, safety controls, and repeatable verification.

**Theory:** Explain mechanism, assumptions, and where this fits in deployment lifecycle.

**Motivation:** Why this matters for reliability, reproducibility, and operations.

**Real-World Example:** Prometheus metrics reveal latency spikes before users complain.

**Visual Explanation:** See figure/code cell below.

**Code Explanation:** Code cell demonstrates concrete implementation details.

**Best Practices:** Use explicit deployment checklist before promoting image.

**Common Mistakes:** Shipping API without health checks, metrics, and rollback plan.

In [3]:
from pathlib import Path
import json

required = [
    'docker-compose.yml',
    'monitoring/prometheus/prometheus.yml',
    'monitoring/grafana/provisioning/datasources/prometheus.yml',
    'monitoring/grafana/provisioning/dashboards/dashboards.yml',
]
for path in required:
    exists = Path(path).exists()
    print(f'{path}:', 'OK' if exists else 'MISSING')

meta = json.loads(Path('models/model_metadata.json').read_text())
print('\nServing model:', meta.get('best_model'))
print('Top metrics:', meta.get('metrics', {}))

docker-compose.yml: OK
monitoring/prometheus/prometheus.yml: OK
monitoring/grafana/provisioning/datasources/prometheus.yml: OK
monitoring/grafana/provisioning/dashboards/dashboards.yml: OK

Serving model: LightGBM
Top metrics: {'MAE': 0.3086374171170888, 'MSE': 0.21458473607446002, 'RMSE': 0.4632329177362723, 'R²': 0.8362459814931062, 'MAPE': 0.18108017014828415}


## Final Checklist
- Training artifacts generated
- API tests pass
- Docker image builds
- Compose stack starts
- Metrics visible in Prometheus
- Dashboard visible in Grafana
- Security baseline reviewed